In [ ]:
include( "../model.jl" )
include( "../visual.jl" )
include( "../cluster.jl" )

using ProgressMeter

# base = "/scratch/mdn7721/vicsek-dynamic-data/system/"
_, base = expfolder( "system" )

In [ ]:
# Hyper parameters.
N = 2048
r = 1.0

# Extract the speed argument.
include( "../0-params/rho-list.jl" )
ρ = 2#ρlist[16]
L = √(N/ρ)

# Speed from user input.
include( "../0-params/spd-list.jl" )
s = 0.5#slist[16]

# Create folderlist.
ηlist = [0.01;0.05:0.05:0.75]
Nη = length( ηlist )

# Print details.
println( "System size:         $(N)" )
println( "Particle speed:      $(round( s, digits=3 ))" )
println( "System-wide density: $(round( ρ, digits=3 ))" )

In [ ]:
# Time till "steady state".
T = 10_000  # TODO: check behavior of order parameter

# Create parameter variable.
paramslist = [
    Dict{Char,Real}( 'N' => N, 'L' => L, 'r' => r, 'v' => s, 'η' => η )
    for η ∈ ηlist];

In [ ]:
# Initialize progress meter.
P = Progress( Nη, "Running to steady state."; dt=1.0 )
Plock = ReentrantLock()

# Initialize state and order parameter lists.
φdata = [Vector{defFloat}( undef, T + 1 ) for _ ∈ 1:Nη]
xlist = [Vector{defFloat}( undef, N ) for _ ∈ 1:Nη]
ylist = [Vector{defFloat}( undef, N ) for _ ∈ 1:Nη]
θlist = [Vector{defFloat}( undef, N ) for _ ∈ 1:Nη]

# Run simulation.
@threads for m ∈ 1:Nη
    # Unpack simulation parameters.
    params = paramslist[m]

    # Initial conditions and order parameter.
    x = L*rand( defFloat, N )
    y = L*rand( defFloat, N )
    θ = 2*π*rand( defFloat, N ) .- π
    φdata[m][1] = order( N, θ )

    # Run the simulation.
    for t ∈ 1:T
        A = adjacency( N, L, r, x, y, θ )
        x, y, θ = vicsek( params, A, x, y, θ )
        φdata[m][t+1] = order( N, θ )
    end

    # Save state variables.
    xlist[m] = x
    ylist[m] = y
    θlist[m] = θ

    # Update progress meter.
    lock( Plock ) do; next!( P ); end
end

In [ ]:
i = 4

# Plot an example order-disorder trend.
plt = plot(; size=(400,250), legend=false )

plot!( plt, φdata[i]; lw=2 )

plot!( plt; xlims=(0,T), ylims=(0,1) )
plot!( plt; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )

In [ ]:
# Plot an example order-disorder trend.
plt = plot(; size=(400,250), legend=false )

plot!( plt, ηlist, mean.( last.( φdata, 5000 ) ); marker=:circ, lw=2 )

plot!( plt; xlims=(0,ηlist[end]), ylims=(0,1) )
plot!( plt; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )

In [ ]:
# Initialize variables for saving.
Tsave = 10_000  # TODO: check behavior of order parameter
φsave = [Vector{defFloat}( undef, Tsave + 1 ) for _ ∈ 1:Nη]

# Set first position to final from initialization.
for m ∈ 1:Nη
    φsave[m][1] = φdata[m][end]
end

# Initialize cluster-related data variables.
clist = [zeros( defInt, N ) for _ ∈ 1:Nη];

In [ ]:
# Initialize new progress meter.
P = Progress( Nη*Tsave, "Running simulations."; dt=1.0 )

# Run simulation from steady state.
@threads for m ∈ 1:Nη
    # Unpack parameters.
    params = paramslist[m]

    # Unpack state variables.
    x = xlist[m];  y = ylist[m];  θ = θlist[m]

    # Run simulation.
    for t ∈ 1:Tsave
        # Compute the current adjacency matrix.
        A = adjacency( N, L, r, x, y, θ )

        # Extract cluster information from adjacency.
        n, _, _ = clusterinfo( N, A )

        # Update cluster distribution count.
        for ni ∈ n;  clist[m][ni] += 1;  end

        # Upadate the position of each agent.
        x, y, θ = vicsek( params, A, x, y, θ )

        # Compute the order parameter.
        φsave[m][t+1] = order( N, θ )

        # Update progress meter.
        next!( P )
    end

    # Save final positions for posterity.
    xlist[m] = x;  ylist[m] = y;  θlist[m] = θ

    # Extract cluster information for the last time step.
    n, _, _ = clusterinfo( N, adjacency( N, L, r, x, y, θ ) )
    for ni ∈ n;  clist[m][ni] += 1;  end
end

# Compute probability map and cluster distribution.
Δclist = [c./sum( c ) for c ∈ clist];

In [ ]:
# Compute the mean order parameter.
μφlist = mean.( φsave )
σφlist = std.(  φsave )

# Extract the critical noise for each (ρ,s) pair.
ηc = criticalnoise( ηlist, μφlist );

In [ ]:
i = Nη-2

# Plot an example order-disorder trend.
plt = plot(; size=(400,250), legend=false )

plot!( plt, φsave[i]; lw=2 )

plot!( plt; xlims=(0,T), ylims=(0,1) )
plot!( plt; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )

In [ ]:
# Plot an example order-disorder trend.
plt = plot(; size=(300,250), legend=false )

plot!( plt, ηlist, μφlist; ribbon=σφlist, marker=:circ, lw=2 )

plot!( plt; xlims=(0,0.8), ylims=(0,1) )
plot!( plt; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )
plot!( plt; framestyle=:box )